# 04 · Retrieval, evaluation and release

## What you will build

The last stage of shipping a language model: deciding where its knowledge should live, measuring it honestly, and pinning down exactly what you released. You will type a TF-IDF retriever from scratch, put it in front of a reader to answer questions from documents, build an evaluation suite with exact match, per-category slices and a regression check between two model versions, and write a release manifest that fingerprints every artefact.

## Learning objectives

- Build a TF-IDF retriever and rank documents by cosine similarity.
- Explain when retrieval beats fine-tuning, and what each costs when facts change.
- Evaluate with normalised exact match, per-category slices and a regression list, not one headline number.
- Record a release so another person can reproduce exactly what was evaluated.

## Dependencies

Tier 1: NumPy and the Python standard library. One optional cell, tagged `comparison`, checks your TF-IDF against scikit-learn's.

### How to work through this notebook

At each **✍️ Your turn**, type the function and run its `practice` check. Once yours matches, the rest of the notebook runs on **your** code.

### A note on scope

The documents describe fictional towns, so no answer depends on knowledge of the real world. The "models" being compared are deliberately simple: a closed-book lookup frozen at an earlier date, and a retrieval-augmented reader. That keeps the evaluation machinery, which is the subject here, in plain view.

## Setup

Run this cell first. It defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours


import hashlib
import json
import re

## 1 · A knowledge base that changes

Each fictional town has three facts: its river, its market day and the year its bridge was built. The knowledge base is the current set of documents. A closed-book model learned an **older snapshot**: since then some market days changed, and some towns were added.

In [ ]:
ORIGINAL = ["Arlo", "Brenna", "Cadwick", "Dunmere", "Elsby", "Farrow", "Galt", "Hollin", "Ivers",
            "Jessop", "Kell", "Lomond", "Marsh", "Nettle", "Orrin", "Pellow", "Quarry", "Rendle"]
NEW = ["Sorrel", "Tarn"]                 # towns added after the closed-book snapshot
SAME_NAME = ["Arlo", "Brenna"]           # also added later: different towns that share a name
RIVERS = ["Venn", "Ash", "Corrow", "Dell", "Esk", "Fenn", "Gyle", "Hart", "Isk", "Jura"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
FEATURES = ["orchards", "mill", "harbour", "clock", "cider", "wool", "ropes", "salt"]

data_rng = np.random.default_rng(4)


def draw_facts():
    return {
        "river": RIVERS[data_rng.integers(len(RIVERS))],
        "market": DAYS[data_rng.integers(len(DAYS))],
        "bridge": str(int(data_rng.integers(1700, 1950))),
    }


facts = {town: draw_facts() for town in ORIGINAL + NEW}
documents = []
for town in ORIGINAL + NEW:
    f = facts[town]
    first, second = data_rng.choice(FEATURES, size=2, replace=False)
    documents.append(
        f"{town}, in the county of Fenwold, lies on the river {f['river']}. The market in {town} opens on "
        f"{f['market']}. The bridge in {town} was built in {f['bridge']}. It is known for its {first} and its {second}."
    )
for town in SAME_NAME:
    f = draw_facts()
    while any(f[key] == facts[town][key] for key in f):  # make every fact differ from the original town's
        f = draw_facts()
    documents.append(
        f"{town}, in the county of Wyke, lies on the river {f['river']}. The market in {town} opens on "
        f"{f['market']}. The bridge in {town} was built in {f['bridge']}."
    )

QUESTION_TEMPLATES = {
    "river": "On which river does {town} lie?",
    "market": "On which day does the market in {town} open?",
    "bridge": "In which year was the bridge in {town} built?",
}
# The evaluation set was written for the Fenwold towns.
questions = [(QUESTION_TEMPLATES[kind].format(town=town), facts[town][kind], kind, town)
             for town in ORIGINAL + NEW for kind in QUESTION_TEMPLATES]
print(len(documents), "documents,", len(questions), "questions")
print(documents[0])
print(documents[-1])

## 2 · TF-IDF from scratch

A retriever needs a way to say "this document is about the same thing as this question". **TF-IDF** scores each word in a document by how often it appears there (term frequency) and how rare it is across the collection (inverse document frequency). Common words such as *the* get little weight; a town's name gets a lot.

With $N$ documents, a word that appears in $\text{df}$ of them gets
$$\text{idf} = \ln\frac{1 + N}{1 + \text{df}} + 1,$$
the smoothed form scikit-learn uses. A document's vector holds term count × idf for every word in the vocabulary, scaled to unit length, so the cosine similarity of two vectors is just their dot product.

### ✍️ Your turn: `tokenize`

Return the list of lowercase word tokens in `text`: everything matched by the regular expression `[a-z0-9]+` after lowercasing.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def tokenize_reference(text):
    return re.findall(r"[a-z0-9]+", text.lower())

In [ ]:
def tokenize(text):
    """Lowercase word tokens, e.g. "Arlo's bridge, 1847." -> ["arlo", "s", "bridge", "1847"]."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
tokenize = practice("tokenize", tokenize, tokenize_reference, cases=[
    ("Arlo's bridge, 1847.",), ("On which river does Arlo lie?",), ("",),
])

### ✍️ Your turn: `fit_tfidf`

Given a list of token lists, return `(vocab, idf)`: `vocab` maps every word to a column index in sorted word order, and `idf` is a float array with the smoothed idf of each word (document frequency counts each document once).

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def fit_tfidf_reference(token_lists):
    vocab = {word: i for i, word in enumerate(sorted({w for tokens in token_lists for w in tokens}))}
    df = np.zeros(len(vocab))
    for tokens in token_lists:
        for word in set(tokens):
            df[vocab[word]] += 1
    n = len(token_lists)
    return vocab, np.log((1 + n) / (1 + df)) + 1

In [ ]:
def fit_tfidf(token_lists):
    """Vocabulary (sorted word -> column) and smoothed idf per word."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
toy = [["the", "river", "venn"], ["the", "market", "opens"], ["the", "river", "ash", "river"]]
fit_tfidf = practice("fit_tfidf", fit_tfidf, fit_tfidf_reference, cases=[(toy,), (toy[:1],)])

### ✍️ Your turn: `tfidf_vectors`

Return a matrix with one row per token list: count of each vocabulary word × its idf, then each row divided by its Euclidean length (rows of all zeros stay zero). Words not in `vocab` are ignored.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def tfidf_vectors_reference(token_lists, vocab, idf):
    matrix = np.zeros((len(token_lists), len(vocab)))
    for row, tokens in enumerate(token_lists):
        for word in tokens:
            if word in vocab:
                matrix[row, vocab[word]] += 1
    matrix *= idf
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms > 0)

In [ ]:
def tfidf_vectors(token_lists, vocab, idf):
    """Unit-length TF-IDF rows, shape (len(token_lists), len(vocab))."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
toy_vocab, toy_idf = fit_tfidf_reference(toy)
tfidf_vectors = practice("tfidf_vectors", tfidf_vectors, tfidf_vectors_reference, cases=[
    (toy, toy_vocab, toy_idf), ([["unknown", "words"]], toy_vocab, toy_idf),
])

### Optional: check against scikit-learn

The cell below builds the same vectors with scikit-learn's `TfidfVectorizer` (smoothed idf, unit length) and compares them with yours. Nothing later depends on it.

In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(token_pattern=r"[a-z0-9]+", lowercase=True)
    reference_matrix = vectorizer.fit_transform(documents).toarray()
    doc_tokens = [tokenize(doc) for doc in documents]
    vocab_check, idf_check = fit_tfidf(doc_tokens)
    mine = tfidf_vectors(doc_tokens, vocab_check, idf_check)
    order = [vectorizer.vocabulary_[word] for word in sorted(vocab_check, key=vocab_check.get)]
    print("largest difference from scikit-learn:", float(np.abs(mine - reference_matrix[:, order]).max()))
except ImportError:
    print("scikit-learn is not installed here; skipping the comparison.")

## 3 · Retrieve, then read

Rank the documents by cosine similarity to the question and keep the top $k$. Then a **reader** extracts the answer from the retrieved text. Here the reader is a simple pattern match on the document's sentences; in a real system it is the language model itself, reading the retrieved passages in its prompt.

### ✍️ Your turn: `top_k`

Return the indices of the `k` rows of `doc_matrix` with the highest dot product with `query_vector`, best first. Break ties by the lower index (hint: `np.argsort(-scores, kind="stable")`).

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def top_k_reference(query_vector, doc_matrix, k):
    scores = doc_matrix @ query_vector
    return [int(i) for i in np.argsort(-scores, kind="stable")[:k]]

In [ ]:
def top_k(query_vector, doc_matrix, k):
    """Indices of the k most similar documents, best first."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
toy_matrix = tfidf_vectors_reference(toy, toy_vocab, toy_idf)
toy_query = tfidf_vectors_reference([["river", "venn"]], toy_vocab, toy_idf)[0]
top_k = practice("top_k", top_k, top_k_reference, cases=[
    (toy_query, toy_matrix, 2), (toy_query, toy_matrix, 3), (np.zeros(len(toy_vocab)), toy_matrix, 2),
])

In [ ]:
doc_tokens = [tokenize(doc) for doc in documents]
vocab, idf = fit_tfidf(doc_tokens)
doc_matrix = tfidf_vectors(doc_tokens, vocab, idf)
PATTERNS = {
    "river": r"lies on the river (\w+)",
    "market": r"opens on (\w+)",
    "bridge": r"was built in (\d+)",
}


def read_answer(kind, passages):
    """The reader: the first passage that states this kind of fact supplies the answer."""
    for passage in passages:
        match = re.search(PATTERNS[kind], passage)
        if match:
            return match.group(1)
    return "I don't know"


def rag_answer(question, kind, k=1):
    query = tfidf_vectors([tokenize(question)], vocab, idf)[0]
    return read_answer(kind, [documents[i] for i in top_k(query, doc_matrix, k)])


question, answer, kind, town = questions[0]
print(question, "->", rag_answer(question, kind), "| expected", answer)

## 4 · Two model versions

- **v1, closed-book.** The facts were learned from a snapshot taken before the latest update: since then four towns changed their market day, and the four newest towns did not exist yet. Asked about those, it answers from stale memory or not at all.
- **v2, retrieval-augmented.** It reads today's documents, so updates reach it as soon as the index is rebuilt, with no retraining. Its weakness is the retriever: the latest index also contains two *different* towns, in the county of Wyke, that share a name with towns in the evaluation set.

In [ ]:
snapshot = {town: dict(f) for town, f in facts.items()}
CHANGED = ["Cadwick", "Galt", "Marsh", "Nettle"]  # market days changed after the snapshot
for town in CHANGED:
    snapshot[town]["market"] = next(d for d in DAYS if d != facts[town]["market"])
for town in NEW:  # towns added after the snapshot
    snapshot.pop(town)


def closed_book_answer(question, kind, town):
    return snapshot.get(town, {}).get(kind, "I don't know")


v1_predictions = [closed_book_answer(q, kind, town) for q, _, kind, town in questions]
v2_predictions = [rag_answer(q, kind) for q, _, kind, _ in questions]
print("v1:", v1_predictions[:3], "\nv2:", v2_predictions[:3])

## 5 · Exact match, normalised

A fair comparison of answers ignores case, surrounding punctuation and articles: "The Venn." and "venn" are the same answer. Normalise both sides, then compare.

### ✍️ Your turn: `exact_match`

Return `True` when `prediction` and `answer` are equal after normalising both: lowercase, remove every character that is not a letter, digit or space, drop the words `a`, `an` and `the`, and join the remaining words with single spaces.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def _normalise_answer(text):
    words = re.sub(r"[^a-z0-9 ]", " ", text.lower()).split()
    return " ".join(word for word in words if word not in {"a", "an", "the"})


def exact_match_reference(prediction, answer):
    return _normalise_answer(prediction) == _normalise_answer(answer)

In [ ]:
def exact_match(prediction, answer):
    """Normalised exact match."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
exact_match = practice("exact_match", exact_match, exact_match_reference, cases=[
    ("The Venn.", "venn"), ("Venn river", "Venn"), ("1847", " 1847 "), ("an  Ash", "ash"), ("Tuesday", "Monday"),
])

## 6 · Slices and regressions

One headline accuracy hides where a model fails. **Slices** report accuracy per category; a **regression list** names every question the old version answered correctly and the new one gets wrong. A release can improve overall and still break something a user relied on.

### ✍️ Your turn: `slice_accuracy`

Given a list of booleans `correct` and a parallel list of `categories`, return a dict mapping each category (in sorted order) to the fraction of its questions answered correctly.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def slice_accuracy_reference(correct, categories):
    return {c: float(np.mean([ok for ok, cat in zip(correct, categories) if cat == c])) for c in sorted(set(categories))}

In [ ]:
def slice_accuracy(correct, categories):
    """Accuracy per category, as {category: fraction}."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
slice_accuracy = practice("slice_accuracy", slice_accuracy, slice_accuracy_reference, cases=[
    ([True, False, True, True], ["a", "a", "b", "b"]), ([False], ["x"]),
])

### ✍️ Your turn: `find_regressions`

Return the indices (in increasing order) where `old_correct` is `True` and `new_correct` is `False`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def find_regressions_reference(old_correct, new_correct):
    return [i for i, (old, new) in enumerate(zip(old_correct, new_correct)) if old and not new]

In [ ]:
def find_regressions(old_correct, new_correct):
    """Questions the old version got right and the new one gets wrong."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
find_regressions = practice("find_regressions", find_regressions, find_regressions_reference, cases=[
    ([True, True, False, True], [True, False, True, False]), ([False, False], [True, True]),
])

In [ ]:
categories = [kind for _, _, kind, _ in questions]
v1_correct = [exact_match(p, a) for p, (_, a, _, _) in zip(v1_predictions, questions)]
v2_correct = [exact_match(p, a) for p, (_, a, _, _) in zip(v2_predictions, questions)]
print(f"overall exact match: v1 {np.mean(v1_correct):.3f}, v2 {np.mean(v2_correct):.3f}")
print("v1 by category:", {k: round(v, 3) for k, v in slice_accuracy(v1_correct, categories).items()})
print("v2 by category:", {k: round(v, 3) for k, v in slice_accuracy(v2_correct, categories).items()})
regressions = find_regressions(v1_correct, v2_correct)
print(f"\n{len(regressions)} regression(s):")
for i in regressions:
    q, a, kind, town = questions[i]
    print(f"  {q}  expected {a!r}, v1 {v1_predictions[i]!r}, v2 {v2_predictions[i]!r}")

**Expected behaviour.** v2 wins overall, and on every changed or new fact. But it regresses on every question about *Arlo* and *Brenna*: the new Wyke towns share those names, and their documents are shorter. TF-IDF divides each document by its length, so the same three mentions of "Arlo" weigh more in the shorter document, which then ranks first. The overall score hides this; the regression list names it.

### 🧪 Try it: retrieve more than one passage

Change `k_try` and re-run the cell. Reading more passages does not fix the regressions: the reader still trusts the first passage, which is about the wrong Arlo. What extra information would let a system tell the two towns apart?

In [ ]:
k_try = 2  # try 1, 2 and 3
answers_k = [rag_answer(q, kind, k=k_try) for q, _, kind, _ in questions]
correct_k = [exact_match(p, a) for p, (_, a, _, _) in zip(answers_k, questions)]
print(f"k={k_try}: exact match {np.mean(correct_k):.3f}, regressions against v1: {len(find_regressions(v1_correct, correct_k))}")

### Retrieval or fine-tuning?

- **Facts that change** (prices, opening days, new products) belong in documents a retriever reads: updating them means re-indexing, not retraining, and the answer can cite its source.
- **Behaviour** (a format, a tone, following a template, a skill) belongs in the weights: fine-tuning or adapters such as LoRA change how the model responds.
- **Live data and actions** (today's weather, a booking) need **tools**: the model calls an API instead of remembering anything.

Many systems use all three. The evaluation suite is what tells you which change actually helped.

## 7 · Pin down what you released

An evaluation number is only meaningful together with exactly what was evaluated. A **release manifest** records a fingerprint of every artefact (base model revision, tokenizer and chat template, adapter weights, retrieval index) beside the evaluation results. Anyone can then check that the model they run is the one that was measured.

### ✍️ Your turn: `fingerprint`

Return the first 16 hexadecimal characters of the SHA-256 hash of `data` (a `bytes` object). Hint: `hashlib.sha256(data).hexdigest()`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def fingerprint_reference(data):
    return hashlib.sha256(data).hexdigest()[:16]

In [ ]:
def fingerprint(data):
    """A short, stable fingerprint of some bytes."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
fingerprint = practice("fingerprint", fingerprint, fingerprint_reference, cases=[
    (b"",), (b"chat template v2",), (np.arange(4.0).tobytes(),),
])

In [ ]:
template = "<|user|>{question}<|assistant|>"
adapter = np.random.default_rng(0).normal(size=(4, 8))  # stands in for LoRA weights
manifest = {
    "base_model": {"name": "tiny-char-lm", "revision": "pretrain-step-1200"},
    "chat_template": {"text": template, "fingerprint": fingerprint(template.encode())},
    "adapter": {"rank": 4, "fingerprint": fingerprint(adapter.tobytes())},
    "retrieval_index": {"documents": len(documents), "fingerprint": fingerprint("\n".join(documents).encode())},
    "evaluation": {
        "questions": len(questions),
        "exact_match": round(float(np.mean(v2_correct)), 3),
        "by_category": {k: round(v, 3) for k, v in slice_accuracy(v2_correct, categories).items()},
        "regressions_against_previous": len(regressions),
    },
}
print(json.dumps(manifest, indent=2))

## 8 · Put it together

In [ ]:
assert np.mean(v2_correct) > np.mean(v1_correct), "retrieval should beat the stale snapshot overall"
changed = [i for i, (_, _, kind, town) in enumerate(questions) if town in CHANGED and kind == "market"]
assert all(v2_correct[i] for i in changed) and not any(v1_correct[i] for i in changed), "updates reach v2 only"
assert all(v2_correct[i] for i, q in enumerate(questions) if q[3] in NEW), "new towns are answered from the index"
assert len(regressions) >= 1, "the suite should surface at least one regression"
assert {questions[i][3] for i in regressions} <= set(SAME_NAME), "every regression comes from the name clash"
assert fingerprint(template.encode()) == manifest["chat_template"]["fingerprint"]
print(f"✅ Retrieval beats the stale model ({np.mean(v2_correct):.3f} vs {np.mean(v1_correct):.3f}), "
      f"and the suite names {len(regressions)} regression(s) an overall score would hide.")

## Exercises

1. **Recall@k.** Write `retrieval_recall(k)`: the fraction of questions whose own town's document is among the top `k`. Plot it for `k = 1…5`.
2. **Disambiguate.** Rewrite the evaluation questions as "On which river does Arlo, in Fenwold, lie?" and re-run the comparison. Does the extra word fix retrieval? Then try the systems approach instead: store each document's county as metadata and filter before ranking.
3. **Abstain.** Make v2 answer "I don't know" when the best retrieval score is below a threshold. How do you choose the threshold, and on which data?
4. **Open question.** Your manifest fingerprints the adapter but not the evaluation questions. Why does fingerprinting the evaluation set matter too, and what goes wrong if it silently changes between releases?

## Common errors and debugging

- **Every answer is "I don't know"**: the reader's pattern does not match; print the retrieved passages.
- **A plausible answer about the wrong entity**: two documents share a name; print the top passages and their scores before trusting the reader.
- **TF-IDF rows not unit length**: divide by the norm after multiplying by idf, and guard against all-zero rows.
- **Exact match too strict or too loose**: normalise both sides the same way, and check what your normaliser removes.
- **A different fingerprint for the "same" file**: whitespace or byte order changed; fingerprint exactly the bytes you ship.

## Summary

Retrieval keeps changing facts out of the weights, so updates need re-indexing rather than retraining, but it brings its own failure modes. An evaluation suite needs more than one number: per-category slices and a regression list show what a release breaks as well as what it fixes. A release manifest ties those results to fingerprints of exactly what was shipped.

## Next

This completes the course. The capstone asks you to package an adapted model with its template, adapter, index and evaluation report. **Vision-Language Models** applies the same build-it-yourself approach to images and text.